## UAV Intrusion Detection Stacked Ensemble
This notebook implements a meta-classifier ensemble using One-Class SVM, Autoencoder, and Local Outlier Factor as described in the Whelan et al. (2020) paper into the UAV Attack Dataset by Jason Whelan, et al.

In [39]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import os

In [40]:
RANDOM_STATE = 1082

base_dir = "../../../Datasets/csv/UAVATTACK/"
file_names = [
    "spoofing-CTGAN-synthetic.csv",
    "spoofing-GaussianCopula-synthetic.csv",
    "spoofing-TVAE-synthetic.csv",
]

In [41]:
for file_name in file_names:
    file_path = os.path.join(base_dir, file_name)
    try:
        df = pd.read_csv(file_path)
        print(f"\n--- Processing {file_name} ---")
    except FileNotFoundError:
        print(f"File not found: {file_path}. Skipping.")
        continue
    except Exception as e:
        print(f"Error loading {file_name}: {e}")
        continue

    # Check required columns
    if not set(['label']).issubset(df.columns):
        print(f"Missing required columns in {file_name}. Skipping.")
        continue

    # Prepare features and labels
    X = df.drop(columns=["label"])
    y = df["label"].astype(int)  # Use existing numeric labels

    # Use benign data to train unsupervised models
    X_train_raw = X[y == 0]
    X_test_raw = X
    y_test = y

    # Normalize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    # PCA for OC-SVM and LOF (85% variance)
    pca_svm_lof = PCA(n_components=0.85, random_state=RANDOM_STATE)
    X_train_pca_svm_lof = pca_svm_lof.fit_transform(X_train_scaled)
    X_test_pca_svm_lof = pca_svm_lof.transform(X_test_scaled)

    # PCA for Autoencoder (95% variance)
    pca_ae = PCA(n_components=0.95, random_state=RANDOM_STATE)
    X_train_pca_ae = pca_ae.fit_transform(X_train_scaled)
    X_test_pca_ae = pca_ae.transform(X_test_scaled)

    # One-Class SVM
    ocsvm = OneClassSVM(nu=0.011, gamma=0.000211)
    ocsvm.fit(X_train_pca_svm_lof)
    ocsvm_pred = np.where(ocsvm.predict(X_test_pca_svm_lof) == -1, 1, 0)

    # Local Outlier Factor
    lof = LocalOutlierFactor(n_neighbors=1000, novelty=True)
    lof.fit(X_train_pca_svm_lof)
    lof_pred = np.where(lof.predict(X_test_pca_svm_lof) == -1, 1, 0)

    # Autoencoder using MLP
    ae = MLPRegressor(hidden_layer_sizes=(3,), activation='relu', max_iter=100, random_state=RANDOM_STATE)
    ae.fit(X_train_pca_ae, X_train_pca_ae)
    recon = ae.predict(X_test_pca_ae)
    recon_error = np.mean((X_test_pca_ae - recon) ** 2, axis=1)
    train_recon = ae.predict(X_train_pca_ae)
    train_error = np.mean((X_train_pca_ae - train_recon) ** 2, axis=1)
    threshold = np.percentile(train_error, 97.2)
    ae_pred = np.where(recon_error > threshold, 1, 0)

    # Stack predictions from the 3 models
    stacked_features = np.vstack([ocsvm_pred, lof_pred, ae_pred]).T

    # Meta-classifier (Random Forest)
    meta_clf = RandomForestClassifier(random_state=RANDOM_STATE)
    meta_clf.fit(stacked_features, y_test)
    stacked_pred = meta_clf.predict(stacked_features)

    # Evaluation
    print(classification_report(
        y_test,
        stacked_pred,
        labels=[0, 1],
        target_names=["benign", "malicious"],
        zero_division=0
    ))


--- Processing spoofing-CTGAN-synthetic.csv ---


C:\Users\antho\miniforge3\envs\sklearn-env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


              precision    recall  f1-score   support

      benign       1.00      0.90      0.95      4169
   malicious       0.93      1.00      0.97      5831

    accuracy                           0.96     10000
   macro avg       0.97      0.95      0.96     10000
weighted avg       0.96      0.96      0.96     10000


--- Processing spoofing-GaussianCopula-synthetic.csv ---


C:\Users\antho\miniforge3\envs\sklearn-env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


              precision    recall  f1-score   support

      benign       0.54      0.97      0.69      4991
   malicious       0.84      0.18      0.30      5009

    accuracy                           0.57     10000
   macro avg       0.69      0.58      0.50     10000
weighted avg       0.69      0.57      0.50     10000


--- Processing spoofing-TVAE-synthetic.csv ---


C:\Users\antho\miniforge3\envs\sklearn-env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


              precision    recall  f1-score   support

      benign       1.00      0.81      0.90      5122
   malicious       0.83      1.00      0.91      4878

    accuracy                           0.90     10000
   macro avg       0.92      0.91      0.90     10000
weighted avg       0.92      0.90      0.90     10000

